# Step 2b: why is the TPU fast (or not)?

Run on a **TPU runtime**. Three short experiments on the same Tesla LSTM:

1. **Compile vs run time.** The first epoch includes XLA compiling the model, so we time it separately.
2. **Batch size sweep.** A TPU's matrix unit works on 128x128 tiles. A 32-unit LSTM with batch 32 leaves most of it idle, so throughput should grow a lot with bigger batches.
3. **Look at what XLA produced.** Print the compiled program for one small function and count the fused kernels, then capture a profile.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"
import time, numpy as np, pandas as pd
import jax, jax.numpy as jnp
import keras
from keras import layers
keras.utils.set_random_seed(42)
print("devices:", jax.devices())

import textwrap
def explain(text):
    print("\n  WHAT THIS MEANS")
    for para in textwrap.dedent(text).strip().split("\n\n"):
        print(textwrap.fill(" ".join(para.split()), width=90, initial_indent="  ", subsequent_indent="  "))
        print()

platform = jax.devices()[0].platform.upper()
explain(f'''
JAX found {len(jax.devices())} {platform} device(s). On a Colab TPU you should see TpuDevice
entries, one per TPU core. If it says CPU, change the runtime type to TPU and restart.
''')

In [ ]:
# Upload TSLA.csv if it's not already in /content
if not os.path.exists("/content/TSLA.csv"):
    from google.colab import files
    up = files.upload()
    os.rename(list(up)[0], "/content/TSLA.csv")

close = pd.read_csv("/content/TSLA.csv", parse_dates=["Date"]).sort_values("Date")["Close"].ffill().to_numpy("float32")
WINDOW = 60
X = np.stack([close[t-WINDOW:t] / close[t-1] - 1 for t in range(WINDOW, len(close))])[..., None].astype("float32")
y = np.array([close[t] / close[t-1] - 1 for t in range(WINDOW, len(close))], dtype="float32")
X_tr, y_tr = X[: int(len(X) * 0.7)], y[: int(len(X) * 0.7)]
print("train windows:", X_tr.shape)

## 1 and 2: compile time and batch size sweep

In [ ]:
def build(units=32):
    m = keras.Sequential([keras.Input((WINDOW, 1)), layers.LSTM(units), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m

rows = []
for units in [32, 256]:
    for batch in [32, 128, 512, 1024]:
        m = build(units)
        t0 = time.perf_counter()
        m.fit(X_tr, y_tr, batch_size=batch, epochs=1, verbose=0)      # includes XLA compile
        first = time.perf_counter() - t0
        t0 = time.perf_counter()
        m.fit(X_tr, y_tr, batch_size=batch, epochs=3, verbose=0)      # already compiled
        steady = (time.perf_counter() - t0) / 3
        rows.append({"units": units, "batch": batch,
                     "first epoch (s)": first, "steady ms/epoch": steady * 1000,
                     "samples/s": len(X_tr) / steady})
        print(rows[-1])

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False, float_format=lambda v: f"{v:.1f}"))

lines = []
for u in sweep["units"].unique():
    sub = sweep[sweep["units"] == u].set_index("batch")
    b_lo, b_hi = sub.index.min(), sub.index.max()
    gain = sub.loc[b_hi, "samples/s"] / sub.loc[b_lo, "samples/s"]
    compile_s = (sub["first epoch (s)"] - sub["steady ms/epoch"] / 1000).mean()
    lines.append(f"{u}-unit LSTM: going from batch {b_lo} to {b_hi} changed throughput "
                 f"{gain:.1f}x ({sub.loc[b_lo, 'samples/s']:,.0f} to {sub.loc[b_hi, 'samples/s']:,.0f} samples/s). "
                 f"Compiling took about {compile_s:.1f}s per new model/batch size.")
if len(sweep["units"].unique()) > 1:
    s32 = sweep[sweep["units"] == 32].set_index("batch")["steady ms/epoch"]
    s256 = sweep[sweep["units"] == 256].set_index("batch")["steady ms/epoch"]
    b = s32.index.max()
    lines.append(f"At batch {b}, the 256-unit model has about 64x more LSTM weights than the 32-unit "
                 f"one but took {s256[b] / s32[b]:.1f}x the time per epoch.")
explain("\n\n".join(lines) + '''

Each training step has a fixed overhead, and a TPU multiplies 128x128 blocks at once. With
batch 32 and 32 units the blocks are mostly empty, so the chip waits. Bigger batches and
bigger layers fill the blocks, so the chip processes more samples per second. If
the gain is large, the original setup was underusing the TPU. If it is small (likely on a
CPU), the hardware was already busy.

An LSTM also runs its 60 time steps one after another, which limits how much work the chip
can do in parallel. That is why LSTMs are a weak fit for TPUs compared with Transformers.
''')

**How to read it:** "first epoch" minus "steady" is roughly the XLA compile cost. If samples/s keeps rising with batch size, the chip was underused at batch 32. Compare the 32-unit and 256-unit rows: a bigger model fills the matrix unit better, so it costs less extra time than you'd expect.

## 3a: what XLA does to a small function

In [ ]:
def layer(x, w, b):
    return jax.nn.tanh(x @ w + b) * 2.0 + 1.0   # matmul + add + tanh + mul + add

x = jnp.ones((1024, 256)); w = jnp.ones((256, 256)); b = jnp.ones((256,))
lowered = jax.jit(layer).lower(x, w, b)
before = lowered.as_text()                 # what JAX hands to XLA
after = lowered.compile().as_text()        # what XLA produced for this chip
entry = after[after.index("ENTRY"):]               # the program the chip actually runs
print("math ops JAX gave XLA:", before.count("stablehlo."))
print("kernels XLA launches :", sum(("fusion(" in l or "custom-call(" in l or " dot(" in l or "convolution(" in l) for l in entry.splitlines()))
print(); print(entry)

n_ops = before.count("stablehlo.")
explain(f'''
JAX turned our 5 math steps (matmul, add, tanh, multiply, add) into {n_ops} basic
operations (called StableHLO) and handed them to XLA. The ENTRY block above is what XLA actually runs on the
{platform}. Lines containing "fusion" are kernels where XLA merged several operations
into one, so the data is read from memory once instead of once per operation. That
merging (fusion) is the main reason jax.jit code is fast, and it's what you would
write by hand with Pallas when XLA's version isn't good enough.
''')

Before XLA there is one op per math step. After compiling, the add, tanh, multiply and add are typically **fused** into the matmul or into one element-wise kernel, so the data goes through memory once instead of five times. That is the main thing XLA does for you.

## 3b: capture a profile

In [ ]:
!pip -q install tensorboard-plugin-profile
m = build(256)
m.fit(X_tr, y_tr, batch_size=512, epochs=1, verbose=0)   # compile first
jax.profiler.start_trace("/content/trace")
m.fit(X_tr, y_tr, batch_size=512, epochs=2, verbose=0)
jax.profiler.stop_trace()
print("trace saved to /content/trace")
explain('''
A profile records what the chip and the host CPU did over time during those 2 epochs. In
the next cell's TensorBoard, open the Profile tab. In trace_viewer, gaps on the TPU rows
mean the TPU was waiting, usually on Python or data input. overview_page gives a single
"device busy" percentage. A low number confirms this model is too small to keep a TPU
busy.
''')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/trace

In TensorBoard open the **Profile** tab, then **trace_viewer** or **overview_page**. Look for how much time the TPU is busy versus idle waiting on the host. Take a screenshot; it's useful evidence for your answer.